In [332]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import pickle
import numpy as np

In [333]:
s1 = pd.read_csv('Data/100_sgn_metric_list_hist_DiffNetFDR_Xia2015.csv')
s2 = pd.read_csv('Data/200_sgn_metric_list_hist_DiffNetFDR_Xia2015.csv')

r1 = pd.read_csv('Data/100_sgn_metric_list_hist_DiffNetFDR_Liu2017.csv')
r2 = pd.read_csv('Data/200_sgn_metric_list_hist_DiffNetFDR_Liu2017.csv')


with open('Data/our_method_metric_list', 'rb') as f:
    our_method_metric_list = pickle.load(f)

runs_precision_list = our_method_metric_list[0]
runs_recall_list = our_method_metric_list[1]
runs_fdr_list = our_method_metric_list[2]
runs_qs_list = our_method_metric_list[3]
runs_time_mean_list = our_method_metric_list[4]
runs_time_se_list = our_method_metric_list[5]
runs_fdr_se_list = our_method_metric_list[6]
runs_recall_se_list = our_method_metric_list[7]

In [334]:
# Construct markdown table for Mean ± SE (sample standard error deviation)

table = f"""
| Dimension | DiffNetFDR_Xia2015 | DiffNetFDR_Liu2017 | DNetEx |
|---|---|---|---|
| 100 | {s1.groupby('alpha').diffnetfinder_sec.agg(['mean']).reset_index()['mean'][0]:.4f} ± {(s1.groupby('alpha').diffnetfinder_sec.agg(['std']).reset_index()['std'][0] / np.sqrt(s1.groupby('alpha').diffnetfinder_sec.agg(['count']).reset_index()['count'][0])):.4f} | {r1.groupby('alpha').diffnetfinder_sec.agg(['mean']).reset_index()['mean'][0]:.4f} ± {(r1.groupby('alpha').diffnetfinder_sec.agg(['std']).reset_index()['std'][0] / np.sqrt(r1.groupby('alpha').diffnetfinder_sec.agg(['count']).reset_index()['count'][0])):.4f} | {runs_time_mean_list[0][0]:.4f} ± {runs_time_se_list[0][0]:.4f} |
| 200 | {s2.groupby('alpha').diffnetfinder_sec.agg(['mean']).reset_index()['mean'][0]:.4f} ± {(s2.groupby('alpha').diffnetfinder_sec.agg(['std']).reset_index()['std'][0] / np.sqrt(s2.groupby('alpha').diffnetfinder_sec.agg(['count']).reset_index()['count'][0])):.4f} | {r2.groupby('alpha').diffnetfinder_sec.agg(['mean']).reset_index()['mean'][0]:.4f} ± {(r2.groupby('alpha').diffnetfinder_sec.agg(['std']).reset_index()['std'][0] / np.sqrt(r2.groupby('alpha').diffnetfinder_sec.agg(['count']).reset_index()['count'][0])):.4f} | {runs_time_mean_list[1][0]:.4f} ± {runs_time_se_list[1][0]:.4f} |
"""

print(table)


| Dimension | DiffNetFDR_Xia2015 | DiffNetFDR_Liu2017 | DNetEx |
|---|---|---|---|
| 100 | 54.3410 ± 0.7124 | 54.0335 ± 0.6962 | 44.7469 ± 0.6246 |
| 200 | 494.9069 ± 6.8294 | 490.8754 ± 6.2727 | 110.1499 ± 1.9713 |



# FDR control

In [335]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# ----- Liu2017 (Blue family) -----
liu_100 = 'rgba(0, 82, 165, 1)'          # Strong blue
liu_100_shade = 'rgba(0, 82, 165, 0.15)'
liu_200 = 'rgba(102, 178, 255, 1)'       # Medium-light blue (still visible)
liu_200_shade = 'rgba(102, 178, 255, 0.15)'

# ----- Xia2015 (Red/Orange family) -----
xia_100 = 'rgba(204, 51, 17, 1)'         # Deep red-orange
xia_100_shade = 'rgba(204, 51, 17, 0.15)'
xia_200 = 'rgba(255, 128, 77, 1)'        # Coral/salmon (visible, not too light)
xia_200_shade = 'rgba(255, 128, 77, 0.15)'

# ----- DNetEx (Green family) -----
dnet_100 = 'rgba(0, 128, 55, 1)'         # Forest green
dnet_100_shade = 'rgba(0, 128, 55, 0.15)'
dnet_200 = 'rgba(102, 194, 115, 1)'      # Medium-light green (still visible)
dnet_200_shade = 'rgba(102, 194, 115, 0.15)'

fig = go.Figure()

# --- p=100 SECTION ---

# Liu2017 p=100
temp_df = r1.groupby('alpha').FDR.mean().reset_index()
grouped = r1.groupby('alpha').FDR.agg(['mean', 'std', 'count']).reset_index()
grouped['sem'] = grouped['std'] / np.sqrt(grouped['count'])

fig.add_trace(go.Scatter(
    x=temp_df.alpha.values, y=temp_df.FDR.values,
    mode='lines+markers', name='DiffNetFDR_Liu2017, p=100',
    line=dict(color=liu_100, width=2.5), 
    marker=dict(color=liu_100, size=7)
))
fig.add_trace(go.Scatter(
    x=pd.concat([grouped['alpha'], grouped['alpha'][::-1]]),
    y=pd.concat([grouped['mean'] + grouped['sem'], (grouped['mean'] - grouped['sem'])[::-1]]),
    fill='toself', fillcolor=liu_100_shade,
    line=dict(color='rgba(255,255,255,0)'), showlegend=False, hoverinfo="skip"
))

# Xia2015 p=100
temp_df2 = s1.groupby('alpha').FDR.mean().reset_index()
grouped2 = s1.groupby('alpha').FDR.agg(['mean', 'std', 'count']).reset_index()
grouped2['sem'] = grouped2['std'] / np.sqrt(grouped2['count'])

fig.add_trace(go.Scatter(
    x=temp_df2.alpha.values, y=temp_df2.FDR.values,
    mode='lines+markers', name='DiffNetFDR_Xia2015, p=100',
    line=dict(color=xia_100, width=2.5), 
    marker=dict(color=xia_100, size=7)
))
fig.add_trace(go.Scatter(
    x=pd.concat([grouped2['alpha'], grouped2['alpha'][::-1]]),
    y=pd.concat([grouped2['mean'] + grouped2['sem'], (grouped2['mean'] - grouped2['sem'])[::-1]]),
    fill='toself', fillcolor=xia_100_shade,
    line=dict(color='rgba(255,255,255,0)'), showlegend=False, hoverinfo="skip"
))

# DNetEx p=100
run_number = 0
fig.add_trace(go.Scatter(
    x=runs_qs_list[run_number], y=runs_fdr_list[run_number],
    mode='lines+markers', name=f'DNetEx, p=100',
    line=dict(color=dnet_100, width=2.5), 
    marker=dict(color=dnet_100, size=7)
))
fig.add_trace(go.Scatter(
    x=list(runs_qs_list[run_number]) + list(runs_qs_list[run_number][::-1]),
    y=list(runs_fdr_list[run_number] + runs_fdr_se_list[run_number]) + list((runs_fdr_list[run_number] - runs_fdr_se_list[run_number])[::-1]),
    fill='toself', fillcolor=dnet_100_shade,
    line=dict(color='rgba(255,255,255,0)'), hoverinfo="skip", showlegend=False
))

# --- p=200 SECTION ---

# Liu2017 p=200
temp_df = r2.groupby('alpha').FDR.mean().reset_index()
grouped = r2.groupby('alpha').FDR.agg(['mean', 'std', 'count']).reset_index()
grouped['sem'] = grouped['std'] / np.sqrt(grouped['count'])

fig.add_trace(go.Scatter(
    x=temp_df.alpha.values, y=temp_df.FDR.values,
    mode='lines+markers', name='DiffNetFDR_Liu2017, p=200',
    line=dict(color=liu_200, width=2.5, dash='dash'), 
    marker=dict(color=liu_200, size=7, symbol='square')
))
fig.add_trace(go.Scatter(
    x=pd.concat([grouped['alpha'], grouped['alpha'][::-1]]),
    y=pd.concat([grouped['mean'] + grouped['sem'], (grouped['mean'] - grouped['sem'])[::-1]]),
    fill='toself', fillcolor=liu_200_shade,
    line=dict(color='rgba(255,255,255,0)'), showlegend=False, hoverinfo="skip"
))

# Xia2015 p=200
temp_df2 = s2.groupby('alpha').FDR.mean().reset_index()
grouped2 = s2.groupby('alpha').FDR.agg(['mean', 'std', 'count']).reset_index()
grouped2['sem'] = grouped2['std'] / np.sqrt(grouped2['count'])

fig.add_trace(go.Scatter(
    x=temp_df2.alpha.values, y=temp_df2.FDR.values,
    mode='lines+markers', name='DiffNetFDR_Xia2015, p=200',
    line=dict(color=xia_200, width=2.5, dash='dash'), 
    marker=dict(color=xia_200, size=7, symbol='square')
))
fig.add_trace(go.Scatter(
    x=pd.concat([grouped2['alpha'], grouped2['alpha'][::-1]]),
    y=pd.concat([grouped2['mean'] + grouped2['sem'], (grouped2['mean'] - grouped2['sem'])[::-1]]),
    fill='toself', fillcolor=xia_200_shade,
    line=dict(color='rgba(255,255,255,0)'), showlegend=False, hoverinfo="skip"
))

# DNetEx p=200
run_number = 1
fig.add_trace(go.Scatter(
    x=runs_qs_list[run_number], y=runs_fdr_list[run_number],
    mode='lines+markers', name=f'DNetEx, p=200',
    line=dict(color=dnet_200, width=2.5, dash='dash'), 
    marker=dict(color=dnet_200, size=7, symbol='square')
))
fig.add_trace(go.Scatter(
    x=list(runs_qs_list[run_number]) + list(runs_qs_list[run_number][::-1]),
    y=list(runs_fdr_list[run_number] + runs_fdr_se_list[run_number]) + list((runs_fdr_list[run_number] - runs_fdr_se_list[run_number])[::-1]),
    fill='toself', fillcolor=dnet_200_shade,
    line=dict(color='rgba(255,255,255,0)'), hoverinfo="skip", showlegend=False
))

# Reference Line
fig.add_trace(go.Scatter(
    x=[min(runs_qs_list[run_number]), max(runs_qs_list[run_number])],
    y=[min(runs_qs_list[run_number]), max(runs_qs_list[run_number])],
    mode='lines', line=dict(color='black', dash='dot', width=1.5), name="f(x) = x"
))

# Layout Updates
fig.update_layout(
    autosize=False,
    xaxis1=dict(tickfont=dict(size=16),showticklabels=True),
    yaxis1=dict(tickfont=dict(size=16),showticklabels=True),
    width=800,
    height=500,
    xaxis_title='Control FDR (q)',
    yaxis_title='Real FDR',
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=50, r=50, t=50, b=50),
    showlegend=True,
    legend=dict(
    x=0.995,
    y=0.01,
    xanchor='right',
    yanchor='bottom',
    bgcolor='rgba(255, 255, 255, 0.8)',
    bordercolor='gray',
    borderwidth=1,
    font=dict(size=13)
    ),
    xaxis=dict(showline=True, linewidth=2, linecolor='black', mirror=True),
    yaxis=dict(showline=True, linewidth=2, linecolor='black', mirror=True),
    font=dict(size=16)
)

# Ensure the output directory exists if it doesn't
import os
if not os.path.exists("Results"):
    os.makedirs("Results")

fig.write_image("Results/fdr-control.pdf")
fig.show()


# Precision-Recall

In [336]:
temp_df = r1.groupby('alpha').aggregate({'NRecall':'mean','FDR':'mean'}).reset_index()
temp_df2 = s1.groupby('alpha').aggregate({'NRecall':'mean','FDR':'mean'}).reset_index()
run_number = 0

# Create figure
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=temp_df.NRecall.values,
        y=1-temp_df.FDR.values,
        mode='lines+markers',
        name='DiffNetFDR_Liu2017, p=100'
    )
)

fig.add_trace(
    go.Scatter(
        x=temp_df2.NRecall.values,
        y=1-temp_df2.FDR.values,
        mode='lines+markers',
        name='DiffNetFDR_Xia2015, p=100'
    )
)

fig.add_trace(
    go.Scatter(
        x=runs_recall_list[run_number],
        y=1-runs_fdr_list[run_number],
        mode='lines+markers',
        name=f'DNetEx, p=100'
    )
)

temp_df = r2.groupby('alpha').aggregate({'NRecall':'mean','FDR':'mean'}).reset_index()
temp_df2 = s2.groupby('alpha').aggregate({'NRecall':'mean','FDR':'mean'}).reset_index()
run_number = 1

fig.add_trace(
    go.Scatter(
        x=temp_df.NRecall.values,
        y=1-temp_df.FDR.values,
        mode='lines+markers',
        name='DiffNetFDR_Liu2017, p=200'
    )
)

fig.add_trace(
    go.Scatter(
        x=temp_df2.NRecall.values,
        y=1-temp_df2.FDR.values,
        mode='lines+markers',
        name='DiffNetFDR_Xia2015, p=200'
    )
)

fig.add_trace(
    go.Scatter(
        x=runs_recall_list[run_number],
        y=1-runs_fdr_list[run_number],
        mode='lines+markers',
        name=f'DNetEx, p=200'
    )
)


# Customize the layout
fig.update_traces(line=dict(width=3))

fig.update_layout(
    xaxis1=dict(tickfont=dict(size=16),showticklabels=True),
    yaxis1=dict(tickfont=dict(size=16),showticklabels=True),
    autosize=False,
    width=800,
    height=500,
    xaxis_title=r"Recall",
    yaxis_title='Precision',
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=50, r=50, t=50, b=50),
    showlegend=True,
    legend=dict(
        x=.995,
        y=0.99,  
        xanchor='right',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.6)',
        bordercolor='gray',
        borderwidth=1
    ),
    xaxis=dict(showline=True, linewidth=2, linecolor='black', mirror=True),
    yaxis=dict(showline=True, linewidth=2, linecolor='black', mirror=True),
    font=dict(size=16)

)

fig.show()

fig.write_image("Results/pr.pdf")


# Power Analysis

In [337]:
pfig = go.Figure()
temp_df = r1.groupby('alpha').TPR.mean().reset_index()
grouped = r1.groupby('alpha').TPR.agg(['mean', 'std', 'count']).reset_index()
grouped['sem'] = grouped['std'] / np.sqrt(grouped['count'])
temp_df2 = s1.groupby('alpha').TPR.mean().reset_index()
grouped2 = s1.groupby('alpha').TPR.agg(['mean', 'std', 'count']).reset_index()
grouped2['sem'] = grouped2['std'] / np.sqrt(grouped2['count'])

run_number = 0

fig = go.Figure()

# DiffNetFDR_Liu2017, p=100 (strong blue)
fig.add_trace(
    go.Scatter(
        x=temp_df.alpha.values,
        y=temp_df.TPR.values,
        mode='lines+markers',
        name='DiffNetFDR_Liu2017, p=100',
        line=dict(color='rgba(0, 82, 165, 1)', width=2.5),
        marker=dict(size=7, color='rgba(0, 82, 165, 1)')
    )
)
# Add the shaded error region
fig.add_trace(go.Scatter(
    x=pd.concat([grouped['alpha'], grouped['alpha'][::-1]]),
    y=pd.concat([grouped['mean'] + grouped['sem'],
                    (grouped['mean'] - grouped['sem'])[::-1]]),
    fill='toself',
    fillcolor='rgba(0, 82, 165, 0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    showlegend=False,
    hoverinfo="skip"
))

# DiffNetFDR_Xia2015, p=100 (deep red-orange)
fig.add_trace(
    go.Scatter(
        x=temp_df2.alpha.values,
        y=temp_df2.TPR.values,
        mode='lines+markers',
        name='DiffNetFDR_Xia2015, p=100',
        line=dict(color='rgba(204, 51, 17, 1)', width=2.5),
        marker=dict(size=7, color='rgba(204, 51, 17, 1)')
    )
)
# Add the shaded error region
fig.add_trace(go.Scatter(
    x=pd.concat([grouped2['alpha'], grouped2['alpha'][::-1]]),
    y=pd.concat([grouped2['mean'] + grouped2['sem'],
                    (grouped2['mean'] - grouped2['sem'])[::-1]]),
    fill='toself',
    fillcolor='rgba(204, 51, 17, 0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    showlegend=False,
    hoverinfo="skip"
))

# DNetEx, p=100 (forest green)
fig.add_trace(
    go.Scatter(
        x=runs_qs_list[run_number],
        y=runs_recall_list[run_number],
        mode='lines+markers',
        name=f'DNetEx, p=100',
        line=dict(color='rgba(0, 128, 55, 1)', width=2.5),
        marker=dict(size=7, color='rgba(0, 128, 55, 1)')
    )
)
# Shaded confidence interval
fig.add_trace(
    go.Scatter(
        x=list(runs_qs_list[run_number]) + list(runs_qs_list[run_number][::-1]),
        y=list(runs_recall_list[run_number] + runs_recall_se_list[run_number]) + list((runs_recall_list[run_number] - runs_recall_se_list[run_number])[::-1]),
        fill='toself',
        fillcolor='rgba(0, 128, 55, 0.15)',
        line=dict(color='rgba(255,255,255,0)'),
        hoverinfo="skip",
        showlegend=False
    )
)



temp_df = r2.groupby('alpha').TPR.mean().reset_index()
grouped = r2.groupby('alpha').TPR.agg(['mean', 'std', 'count']).reset_index()
grouped['sem'] = grouped['std'] / np.sqrt(grouped['count'])
temp_df2 = s2.groupby('alpha').TPR.mean().reset_index()
grouped2 = s2.groupby('alpha').TPR.agg(['mean', 'std', 'count']).reset_index()
grouped2['sem'] = grouped2['std'] / np.sqrt(grouped2['count'])

run_number = 1

# DiffNetFDR_Liu2017, p=200 (medium-light blue, dashed, square markers)
fig.add_trace(
    go.Scatter(
        x=temp_df.alpha.values,
        y=temp_df.TPR.values,
        mode='lines+markers',
        name='DiffNetFDR_Liu2017, p=200',
        line=dict(color='rgba(102, 178, 255, 1)', width=2.5, dash='dash'),
        marker=dict(size=7, color='rgba(102, 178, 255, 1)', symbol='square')
    )
)
# Add the shaded error region
fig.add_trace(go.Scatter(
    x=pd.concat([grouped['alpha'], grouped['alpha'][::-1]]),
    y=pd.concat([grouped['mean'] + grouped['sem'],
                    (grouped['mean'] - grouped['sem'])[::-1]]),
    fill='toself',
    fillcolor='rgba(102, 178, 255, 0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    showlegend=False,
    hoverinfo="skip"
))

# DiffNetFDR_Xia2015, p=200 (coral/salmon, dashed, square markers)
fig.add_trace(
    go.Scatter(
        x=temp_df2.alpha.values,
        y=temp_df2.TPR.values,
        mode='lines+markers',
        name='DiffNetFDR_Xia2015, p=200',
        line=dict(color='rgba(255, 128, 77, 1)', width=2.5, dash='dash'),
        marker=dict(size=7, color='rgba(255, 128, 77, 1)', symbol='square')
    )
)
# Add the shaded error region
fig.add_trace(go.Scatter(
    x=pd.concat([grouped2['alpha'], grouped2['alpha'][::-1]]),
    y=pd.concat([grouped2['mean'] + grouped2['sem'],
                    (grouped2['mean'] - grouped2['sem'])[::-1]]),
    fill='toself',
    fillcolor='rgba(255, 128, 77, 0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    showlegend=False,
    hoverinfo="skip"
))

# DNetEx, p=200 (medium-light green, dashed, square markers)
fig.add_trace(
    go.Scatter(
        x=runs_qs_list[run_number],
        y=runs_recall_list[run_number],
        mode='lines+markers',
        name=f'DNetEx, p=200',
        line=dict(color='rgba(102, 194, 115, 1)', width=2.5, dash='dash'),
        marker=dict(size=7, color='rgba(102, 194, 115, 1)', symbol='square')
    )
)
# Shaded confidence interval
fig.add_trace(
    go.Scatter(
        x=list(runs_qs_list[run_number]) + list(runs_qs_list[run_number][::-1]),
        y=list(runs_recall_list[run_number] + runs_recall_se_list[run_number]) + list((runs_recall_list[run_number] - runs_recall_se_list[run_number])[::-1]),
        fill='toself',
        fillcolor='rgba(102, 194, 115, 0.15)',
        line=dict(color='rgba(255,255,255,0)'),
        hoverinfo="skip",
        showlegend=False
    )
)

fig.update_layout(
    autosize=False,
    xaxis1=dict(tickfont=dict(size=16),showticklabels=True),
    yaxis1=dict(tickfont=dict(size=16),showticklabels=True),
    width=800,
    height=500,
    xaxis_title='Control FDR (q)',
    yaxis_title='Recall',
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=50, r=50, t=50, b=50),
    showlegend=True,
    legend=dict(
        x=0.005,
        y=0.99,  
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.8)',
        bordercolor='gray',
        borderwidth=1,
        font=dict(size=13)
    ),
    xaxis=dict(showline=True, linewidth=2, linecolor='black', mirror=True),
    yaxis=dict(showline=True, linewidth=2, linecolor='black', mirror=True),
    font=dict(size=16)
)

fig.write_image("Results/recall-control.pdf")
fig.show()
